In [ ]:
%%capture
%cd ~/repos/PredUCE/
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl

from make_clinical_dataset.shared.constants import ROOT_DIR
from preduce.emerg.config import EMBED_COLS, META_COLS
from preduce.emerg.prep import Preparer, Splitter, build_features, load_data, split_columns

EMB_MODEL = "PubMedBERT"
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"
DATA_PATH = f'{DATA_DIR}/processed/treatment_centered_data.parquet'
DATE_PATH = f'{DATA_DIR}/processed/treatment_centered_dates.parquet'
NOTE_PATH = f'{DATA_DIR}/interim/subsets/clinic_visits_prior_to_treatment/notes.parquet'

TEXT_PATH = f'{DATA_DIR}/interim/embedding/ed_risk_summary.parquet'
EMB_PATH = f'{DATA_DIR}/interim/embedding/{EMB_MODEL}'

In [ ]:
df = load_data(DATA_PATH, NOTE_PATH, TEXT_PATH)
df = build_features(df)

targ_cols = [col for col in df.columns if col.startswith("target")]
embed_cols = EMBED_COLS
meta_cols = META_COLS

# Split the data into train, valid, test set
splitter = Splitter()
train_df, valid_df, test_df = splitter.split_data(
    df, split_date="2022-01-01", visit_col="assessment_date", 
)

# Transform the three datasets BASED ON the train set
preparer = Preparer(exclude_cols=embed_cols+meta_cols+targ_cols)
train_df = preparer.fit_transform(train_df)
valid_df = preparer.transform(valid_df)
test_df = preparer.transform(test_df)

# Combine the datasets back with split indicator
df = pl.concat([
    train_df.with_columns(pl.lit("train").alias("split")),
    valid_df.with_columns(pl.lit("valid").alias("split")),
    test_df.with_columns(pl.lit("test").alias("split")),
])

# Split data into input tabular features, input embedding features, output targets, meta info
data = split_columns(df, embed_cols, meta_cols, targ_cols)
tab_feats, embed_feats, targs, meta = data["X_tabular"], data["X_embedding"], data["y"], data["meta"]